In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
# uncomment to use hosted db
#del os.environ["BIRDDOG_USE_LOCAL_NOCODB"]
if os.environ.get("BIRDDOG_USE_LOCAL_NOCODB"):
    print("using local db")
else:
    print("using AWS db")

using local db


In [3]:
from datetime import datetime

from birddog.database import Database
from birddog.database_dashboard import (
    get_doc_id_by_process_code,
    assign_docs_to_pages,
    split_doc_list_by_code,
)

2026-05-06 11:09:40,014 [INFO] Using local nocodb api: http://localhost:8080


In [4]:
from contextlib import contextmanager
import time

@contextmanager
def timer(label="Elapsed"):
    start = time.perf_counter()
    try:
        yield
    finally:
        print(f"{label}: {time.perf_counter() - start:.3f}s")

In [5]:
db = Database()

2026-05-06 11:09:40,043 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  localhost:api                         20.00   104.00    39.00       0.00           24


In [6]:
with timer():
    dids = get_doc_id_by_process_code(db, ["P1", "P2", "FX"])

1000
2000
2662
Elapsed: 0.238s


In [7]:
len(dids)

2662

In [9]:
all_dids = list(dids.keys())

In [10]:
with timer():
    dm, pm = assign_docs_to_pages(db, all_dids)

2026-05-06 11:09:46,795 [INFO] _make_doc_map: loading doc records (2662)
2026-05-06 11:09:57,816 [INFO] _make_page_tree: loading owning page records (2638)
2026-05-06 11:10:09,974 [INFO] _make_page_tree: loading parent page records (176)
2026-05-06 11:10:10,744 [INFO] _make_page_tree: loading parent page records (153)
2026-05-06 11:10:11,446 [INFO] _make_page_tree: loading parent page records (3)
Elapsed: 24.737s


In [11]:
sum=0
for p, v in pm.items():
    #if v['level'] not in ('case', 'volume'):
    if v['level'] == 'opus':
        if not v.get('assigned_parent') and v.get('assigned_docs', []):
            print(f"{p}: {v['label']} - no upward assigned doc")
        sum += len(v.get('assigned_docs', []))
        print(f"{p}: {v['label']} ({v.get('level', '')}), assigned parent={v.get('assigned_parent')}, num_docs={len(v.get('assigned_docs', []))}")
sum

8705: DAKIRO-D/404/1 (opus), assigned parent=7231, num_docs=10
8197: DAKIRO-D/78/4 (opus), assigned parent=7117, num_docs=7
3590: DAVIO-D/793/1 (opus), assigned parent=5128, num_docs=3
3078: DAVIO-D/722/1 (opus), assigned parent=2795, num_docs=10
3596: DAVIO-D/904/24 (opus), assigned parent=4803, num_docs=3
6669: DAKIRO-D/242/1 (opus), assigned parent=6429, num_docs=7
5140: DAVIO-D/904/14 (opus), assigned parent=4803, num_docs=4
8212: DAKIRO-D/212/1 (opus), assigned parent=6891, num_docs=1
5656: DAVIO-D/37/1 (opus), assigned parent=3469, num_docs=1
9245: DAKIRO-D/20/1 (opus), assigned parent=6302, num_docs=3
7200: DAKIRO-D/403/1 (opus), assigned parent=6123, num_docs=49
8738: DAKIRO-D/547/1 (opus), assigned parent=6873, num_docs=11
4130: DAVIO-D/9/1 (opus), assigned parent=4421, num_docs=5
2599: DAVIO-D/186/1 (opus), assigned parent=4502, num_docs=3
9259: DAKIRO-D/719/1 (opus), assigned parent=6607, num_docs=23
5166: DAVIO-D/194/1 (opus), assigned parent=4858, num_docs=16
49: DAVIO-R/R

2722

In [12]:
#pm = split_doc_assignments_by_code(pm, dids)

In [13]:
#for p, v in pm.items():
    #if v['level'] not in ('case', 'volume'):
    #if v['level'] == 'opus':
        #if not v.get('assigned_parent') and v.get('assigned_docs', []):
        #    print(f"{p}: {v['label']} - no upward assigned doc")
        #sum += len(v.get('assigned_docs', []))
        #assigned_docs = v.get("assigned_docs", {})
        #if assigned_docs:
        #    print(f"{p}: {v['label']} ({v.get('level', '')}), assigned parent={v.get('assigned_parent')}")
        #    for k, l in assigned_docs.items():
        #        print(f"    {k}: {len(l)}")

In [14]:
def _scan_opus_summary(db, codes):
    fields = [ f"{code}" for code in codes ]
    fields.extend(["page", "label", "url"])
    #print(fields)

    cursor = None
    result = []
    while True:
        records, cursor = db.scan("OpusSummary", cursor=cursor, limit=500, fields=fields)
        result.extend(records)
        if not records or not cursor:
            break

    for rec in result:
        for code in codes:
            doc_ids = rec[code]
            if not isinstance(doc_ids, list):
                rec[code] = db.get_links("OpusSummary", code, rec["Id"])
            else:
                rec[code] = [ d["Id"] for d in doc_ids ]
        page_id = rec["page"]
        if isinstance(page_id, dict):
            rec["page"] = page_id["Id"]
    return result

In [29]:
def update_opus_summary(page_map, doc_code_map):
    codes = { code for code in doc_code_map.values() }

    opus_summary_map = {}
    records_to_delete = []
    for rec in _scan_opus_summary(db, codes):
        linked_page_id = rec.get("page")
        # each summary record must uniquely link to a valid page
        if linked_page_id and linked_page_id not in opus_summary_map:
            opus_summary_map[linked_page_id] = rec
        else:
            records_to_delete.append(rec)
    #print(opus_summary_map)

    records_to_create = set()
    records_to_update = set()

    for page_id, page in page_map.items():
        #if v['level'] not in ('case', 'volume'):
        if page['level'] == 'opus':
            #print(
            #    f"{page_id}: {page['label']} ({page.get('level', '')}),"
            #    f" assigned parent={page.get('assigned_parent')},"
            #    f" num_docs={len(page.get('assigned_docs', []))}")
            summary_record = opus_summary_map.get(page_id)
            if summary_record:
                # known opus page - check if update needed
                assigned_docs = split_doc_list_by_code(
                    page.get('assigned_docs', []), doc_code_map)
                for code, doc_list in assigned_docs:
                    doc_link_field = f"{code}"
                    summary_doc_list = summary_record.get(doc_link_field, [])
                    if set(doc_list) != set(summary_doc_list):
                        summary_record[doc_link_field] = doc_list
                        records_to_update.add(page_id)
            else:
                # new page to be added
                records_to_create.add(page_id)

    if records_to_create:
        records = [{ 
            "url": page_map[page_id]["url"], 
            "label": page_map[page_id]["label"]
        } for page_id in records_to_create]
        print(f"create: {len(records)} records")
        #print(records)
        record_ids = db.write("OpusSummary", records)
        for rec_id, page_id in zip(record_ids, records_to_create):
            print(rec_id, page_id)
            assigned_docs = split_doc_list_by_code(
                page_map[page_id].get('assigned_docs', []), doc_code_map)
            db.create_links("OpusSummary", "page", rec_id, page_id)
            for code, doc_ids in assigned_docs.items():
                if doc_ids:
                    doc_link_field = f"{code}"
                    print(f"{page_id}: {page_map[page_id]['label']}, create doc links: {code}, {len(doc_ids)}")
                    db.create_links("OpusSummary", doc_link_field, rec_id, doc_ids)

    return opus_summary_map, records_to_create, records_to_update, records_to_delete 
                

In [30]:
with timer():
    osm, to_create, to_update, to_delete = update_opus_summary(pm, dids)

2026-05-06 12:36:36,165 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  localhost:api                         30.00     0.75    39.00       0.00           24
Elapsed: 18.711s


In [31]:
to_create

set()

In [32]:
to_update

{49,
 125,
 182,
 314,
 442,
 580,
 747,
 755,
 790,
 848,
 1173,
 1294,
 1417,
 1442,
 1517,
 1763,
 2038,
 2161,
 2248,
 2427,
 2438,
 2538,
 2548,
 2599,
 2635,
 2668,
 2707,
 2711,
 2732,
 2736,
 2764,
 2783,
 2835,
 2871,
 2963,
 3018,
 3078,
 3163,
 3175,
 3291,
 3311,
 3318,
 3416,
 3558,
 3590,
 3596,
 3637,
 3664,
 3681,
 3779,
 3823,
 3824,
 3901,
 3927,
 4034,
 4054,
 4075,
 4130,
 4165,
 4243,
 4264,
 4327,
 4335,
 4443,
 4444,
 4447,
 4494,
 4537,
 4539,
 4554,
 4576,
 4732,
 4735,
 4779,
 4784,
 4818,
 4852,
 4976,
 4994,
 5004,
 5140,
 5166,
 5203,
 5237,
 5255,
 5275,
 5288,
 5291,
 5293,
 5386,
 5456,
 5462,
 5463,
 5507,
 5516,
 5606,
 5656,
 5731,
 5845,
 5888,
 5933,
 6035,
 6062,
 6067,
 6235,
 6237,
 6416,
 6418,
 6489,
 6501,
 6518,
 6535,
 6576,
 6580,
 6669,
 6752,
 6759,
 6764,
 6810,
 6816,
 6905,
 6911,
 6931,
 6947,
 7115,
 7131,
 7164,
 7200,
 7240,
 7362,
 7370,
 7460,
 7556,
 7643,
 7650,
 7666,
 7775,
 7820,
 8023,
 8104,
 8106,
 8117,
 8119,
 8197,
 82

In [33]:
to_delete

[]